In [1]:
config_path = "config/params.yaml"
mode = "default"

In [2]:
# Parameters
config_path = "config/params.yaml"


# 01 — Unduh GH Archive

**Tujuan:** mengunduh 25 jam GH Archive secara immutable dan menilai kualitas tangkapan tiap jam.

In [3]:
import json
from concurrent.futures import ProcessPoolExecutor

import pandas as pd

from ghremoval.config import load_context
from ghremoval.ingest.gharchive import compare_with_local, download_hour, hour_quality
from ghremoval.quality import Checks
from ghremoval.report import save_table
from ghremoval.runlog import RunLog

ctx = load_context(config_path)
ctx.paths.ensure()
run = RunLog("01_ingest_gharchive", ctx.paths.run_log, ctx.paths.root)
checks = Checks()
hours = ctx.all_hours()

## 1. Unduh

In [4]:
entries = [download_hour(ctx, hour) for hour, _ in hours]
print("file:", len(entries), "total GB:", round(sum(e["bytes"] for e in entries) / 1e9, 2))

file: 25 total GB: 2.41


## 2. Kualitas tiap jam

In [5]:
paths = [ctx.paths.raw / f"{hour}.json.gz" for hour, _ in hours]
with ProcessPoolExecutor(max_workers=4) as pool:
    quality = list(pool.map(hour_quality, paths))
table = pd.DataFrame([{"hour_id": hour, "week": week, **q} for (hour, week), q in zip(hours, quality)])
save_table(table, "tab_hour_quality", "Capture quality per hour")
table[["hour_id", "week", "n_lines", "n_bad", "n_minutes", "minute_min", "minute_median", "minute_max", "minutes_below_half_median"]]

,hour_id,week,n_lines,n_bad,n_minutes,minute_min,minute_median,minute_max,minutes_below_half_median
0,2024-04-01-15,dev,272191,0,60,4131,4510.0,4904,0
1,2024-04-09-3,w0,210306,0,60,25,3551.0,4238,1
2,2024-04-09-15,w0,280409,0,60,3381,4648.0,5198,0
3,2024-04-11-3,w0,208923,0,60,3210,3425.5,4040,0
4,2024-04-11-15,w0,265679,0,60,4005,4397.5,4805,0
5,2024-04-13-3,w0,145220,0,60,1991,2340.0,3692,0
6,2024-04-13-15,w0,193562,0,60,2672,3176.5,4095,0
7,2024-04-16-3,w1,176054,0,60,2542,2896.5,3443,0
8,2024-04-16-15,w1,256809,0,60,2388,4296.5,4874,0
9,2024-04-18-3,w1,187898,0,60,2698,3098.5,3823,0


## 3. Jam dev terhadap file lokal pengguna

In [6]:
comparison = compare_with_local(ctx.paths.raw / f"{ctx.p.data['dev_hour']}.json.gz", ctx.paths.root / ctx.p.paths["local_dev_file"])
print(comparison)

{'match': True, 'gz_n_lines': 272191, 'gz_first_created_at': '2024-04-01T15:00:00Z', 'gz_last_created_at': '2024-04-01T15:59:59Z', 'local_n_lines': 272191, 'local_first_created_at': '2024-04-01T15:00:00Z', 'local_last_created_at': '2024-04-01T15:59:59Z'}


## Ringkasan & cek

In [7]:
checks.add("Q01a", len(entries) == 25, f"{len(entries)} file")
checks.add("Q01b", (table["n_minutes"] == 60).all(), "tiap jam memuat 60 menit")
violations = table.loc[table["minutes_below_half_median"] > 0, ["hour_id", "minutes_below_half_median"]]
print("jam dengan menit di bawah 50% median:\n", violations.to_string(index=False) if len(violations) else "tidak ada")
flags = {row.hour_id: int(row.minutes_below_half_median) for row in violations.itertuples()}
(ctx.paths.results / "hour_quality_flags.json").write_text(json.dumps(flags, indent=2) + "\n")
checks.add("Q02", True, f"{len(flags)} jam ditandai, tercatat di results/hour_quality_flags.json")
checks.add("Q03", comparison["match"], "jam dev cocok dengan file lokal")
checks.raise_if_failed()
print(run.finish(n_hours=len(entries), n_events=int(table["n_lines"].sum())))

[LULUS] Q01a: 25 file
[LULUS] Q01b: tiap jam memuat 60 menit
jam dengan menit di bawah 50% median:
      hour_id  minutes_below_half_median
2024-04-09-3                          1
[LULUS] Q02: 1 jam ditandai, tercatat di results/hour_quality_flags.json
[LULUS] Q03: jam dev cocok dengan file lokal
{'notebook': '01_ingest_gharchive', 'finished_at_utc': '2026-11-10T05:49:38+00:00', 'git_commit': 'bdf6ff7', 'duration_seconds': 26.9, 'counts': {'n_hours': 25, 'n_events': 5291894}}
